In [3]:
! pip install openai

In [4]:
import os
from openai import OpenAI

In [5]:
from openai import OpenAI
from google.colab import userdata

# Get the Groq API key from Colab's secrets manager
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Ensure the API key is not None
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab secrets. Please add it.")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

response = client.responses.create(
    input="Explain the importance of fast language models",
    model="openai/gpt-oss-20b",
)
print(response.output_text)

**Why speed matters for language models (LMs) – a quick‑look guide**

| **Why it matters** | **What it buys you** | **Real‑world examples** |
|--------------------|----------------------|-------------------------|
| **Low latency** – the time from query to answer | Instant‑feedback UX, conversational agents, voice‑assistant responsiveness | Siri, Alexa, real‑time translation apps |
| **High throughput** – many queries per second | Serving millions of users, large‑scale search engines, real‑time data pipelines | Google search, OpenAI API, BERT‑based search ranking |
| **Hardware efficiency** – fewer FLOPs or less memory | Lower power, cheaper GPUs/TPUs, edge‑device deployment | On‑device mobile translation, in‑car LLM for driver assistance |
| **Cost‑effectiveness** | Fewer compute credits, smaller infrastructure footprint | Cloud budgets, start‑up scaling, pay‑as‑you‑go APIs |
| **Environment** | Reduced energy consumption → lower carbon footprint | Green AI initiatives |

---

## 1. C

In [8]:
! pip install langchain faiss-cpu langchain_community tiktoken sentence-transformers

In [10]:
#step 1 load document
from langchain_community.document_loaders import TextLoader
loader=TextLoader('/content/school_dataset_qa.txt')
documents=loader.load()

In [11]:
# step 2 create the embedding + vector DB
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize the HuggingFace embeddings
# Using a common open-source model; ensure 'sentence-transformers' is installed
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# Create the vector database from your pre-loaded documents
vector_db = FAISS.from_documents(documents, embeddings)

/tmp/ipykernel_1227/4207546490.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
# #step 3 Retrieval + Generation
def rag_query(query):
    # Retrieve the top 3 most relevant document chunks based on the query
    docs = vector_db.similarity_search(query, k=3)

    # Combine the content of the retrieved documents into a single context string
    context = " ".join([doc.page_content for doc in docs])

    # Generate the response using OpenAI's chat completions API
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile", # Updated model to the user-specified Groq model
        messages=[
            {"role": "system", "content": "use the provided context to answer the question"},
            {"role": "user", "content": f"context: {context}\n\nquery: {query}"}
        ]
    )

    return response.choices[0].message.content

In [13]:
print(rag_query("What are the school timings?"))

The school timings are not specified in the provided data. However, other information such as admissions, documents required, tuition fees, attendance requirements, library resources, transport services, and support services are available.

If you'd like to know more about a specific topic, please let me know and I can try to provide more information based on the available data. 

Here's a code snippet that attempts to find relevant information:

```python
def find_relevant_info(query, dataset):
    for item in dataset:
        if query.lower() in item["text"].lower():
            return item["text"]
    return "No relevant information found."

query = "What are the school timings?"
print(find_relevant_info(query, data))
```

When you run this code, it will output: "No relevant information found." because the school timings are not mentioned in the provided data.


In [14]:
import warnings
warnings.filterwarnings('ignore')
!pip install bitsandbytes accelerate transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 7.1 MB/s eta 0:00:00


In [15]:
# #step 1 : Load model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import bitsandbytes as bnb

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# #define the bitsandbytes config for 8 bit quantization
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [16]:
# #step 2 : Apply LoRA(PEFT)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# #preapre the model for k bit trining
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.5,
    bias="none",
    task_type="CAUSAL_LM"
)
model=get_peft_model(model,lora_config)


In [17]:
from datasets import Dataset

data = [
    {
        "text": "Q: What is the name of the school?\nA: The name of the school is Green Valley Higher Secondary School."
    },
    {
        "text": "Q: What classes does the school offer?\nA: The school offers education from Grade 1 to Grade 12."
    },
    {
        "text": "Q: Is Green Valley School co-educational?\nA: Yes, Green Valley School is a co-educational institution."
    },
    {
        "text": "Q: When are admissions open?\nA: Admissions are open from April to June each year."
    },
    {
        "text": "Q: What documents are required for admission?\nA: Students must submit previous academic records and a birth certificate."
    },
    {
        "text": "Q: How are tuition fees paid?\nA: Tuition fees are paid quarterly."
    },
    {
        "text": "Q: Is there a penalty for late fee payment?\nA: Yes, late payments may incur a penalty charge."
    },
    {
        "text": "Q: What is the minimum attendance requirement?\nA: Students must maintain at least 75% attendance to appear for final examinations."
    },
    {
        "text": "Q: How many books are available in the school library?\nA: The library contains over 15,000 books, journals, and digital resources."
    },
    {
        "text": "Q: Does the school provide transport services?\nA: Yes, school buses operate across major city routes with GPS tracking."
    },
    {
        "text": "Q: Are transport services available for students?\nA: Yes, transport services are available through school buses."
    },
    {
        "text": "Q: What student support services are available?\nA: Academic counseling and career guidance are available for all students."
    },
    {
        "text": "Q: What are the school timings from Monday to Friday?\nA: School timings are 8:30 AM to 4:00 PM from Monday to Friday."
    },
    {
        "text": "Q: What are the school timings on Saturday?\nA: School timings on Saturday are 8:30 AM to 12:30 PM."
    },
    {
        "text": "Q: Is the school open on Sunday?\nA: No, the school is closed on Sundays."
    },
    {
        "text": "Q: How are student records maintained?\nA: Student records are securely maintained and shared only with authorized personnel."
    },
    {
        "text": "Q: Does the school protect student data?\nA: Yes, student records are securely maintained."
    },
    {
        "text": "Q: Are scholarships available for students?\nA: Yes, merit-based and need-based scholarships are available."
    },
    {
        "text": "Q: What types of scholarships are offered?\nA: The school offers merit-based and need-based scholarships."
    },
    {
        "text": "Q: Who can access student records?\nA: Student records can only be shared with authorized personnel."
    }
]

dataset = Dataset.from_list(data)

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 20
})


In [18]:
# step 4: tokenization
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding=True)


tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [20]:
#step 5: Training
from transformers import Trainer , TrainingArguments
#add labels to the tokenized_dataset for causal language mdelling
def add_labels_to_dataset(examples):
  examples['labels']=examples['input_ids']
  return examples
tokenized_dataset=tokenized_dataset.map(add_labels_to_dataset,batched=True)
training_args=TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=50
)
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
trainer.train()

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Step,Training Loss
10,2.165817


TrainOutput(global_step=10, training_loss=2.1658166885375976, metrics={'train_runtime': 551.0975, 'train_samples_per_second': 0.073, 'train_steps_per_second': 0.018, 'total_flos': 6971920293888.0, 'train_loss': 2.1658166885375976, 'epoch': 2.0})

In [28]:
def generate_response(prompt):
    # Put the model in evaluation mode
    model.eval()
    # Dynamically get the model's device
    model_device = model.device if hasattr(model, 'device') else next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

    # Generate tokens, ensure no `do_sample` is set unless specified, to keep generation deterministic
    # Reducing max_new_tokens to prevent timeouts on CPU
    outputs = model.generate(**inputs, max_new_tokens=20)

    # Decode with skip_special_tokens=True for cleaner output
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [29]:
print(generate_response("Q: What are the school timings?\nA:"))

[transformers] Both `max_new_tokens` (=20) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What are the school timings?
A: The school timings are from 8:30 am to 3:00 pm.
